<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/lessons/lesson_49_docker/note_lesson_49_docker_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐳 Урок 49 — Docker

| Крок | Що робимо |
|---|---|
| 0 | `news_hub` уроку 49: `Dockerfile` і `.dockerignore` |
| 1 | читаємо `Dockerfile` як дані (вправа 1) |
| 2 | `.dockerignore`: що потрапить в образ (вправи 2–3) |
| 3 | кеш шарів: чому порядок важливий (вправа 4) |
| 4 | секрети й змінні середовища (вправи 5–6) |
| 5 | сигнали: чому `docker stop` чекає 10 секунд (вправа 7) |
| 6 | знайди помилку: лінтер Dockerfile (вправа 8) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам.

У Colab немає Docker. Тому тут — те, що Docker робить «усередині» і що можна перевірити чистим Python: правила `.dockerignore`, кеш шарів, змінні середовища, сигнали Linux. Команди Docker з реальним виводом — у кінці ноутбука і в книзі курсу: [Урок 49](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_48/). Повтори їх локально, коли встановиш Docker Desktop.

---
## 0. Проєкт

Клітинка завантажує `news_hub` уроку 49 (у Colab). Залежності не потрібні: усе в ноутбуці — стандартна бібліотека Python.

In [ ]:
import os
import subprocess

LESSON = "module_5/lessons/lesson_49_docker"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))
os.chdir("news_hub")

from pathlib import Path

DOCKERFILE = Path("Dockerfile").read_text(encoding="utf-8")
DOCKERIGNORE = Path(".dockerignore").read_text(encoding="utf-8").splitlines()
print(DOCKERFILE)

---
## 1. Dockerfile як дані

Кожен рядок `Dockerfile` — **інструкція** (`FROM`, `COPY`, `RUN`, …) і її аргументи. Рядок, що закінчується на `\`, продовжується на наступному; `#` на початку — коментар.

### Вправа 1

Напиши `parse_dockerfile(text)` → список пар `(ІНСТРУКЦІЯ, аргументи)`: інструкція — великими літерами, продовження рядків склеєні через пробіл, коментарі й порожні рядки пропущені.

In [ ]:
def parse_dockerfile(text: str) -> list[tuple[str, str]]:
    # YOUR CODE HERE
    pass


steps = parse_dockerfile(DOCKERFILE)
for number, (name, args) in enumerate(steps, 1):
    print(f"{number:2}. {name:<11} {args[:80]}")

names = [name for name, _ in steps]
assert names[0] == "FROM" and names[-1] == "CMD"
assert steps[names.index("HEALTHCHECK")][1].startswith("--interval=30s --timeout=5s")     # склеєно з \\
assert names.index("COPY") < names.index("RUN")                        # спершу requirements.txt, потім pip
assert ("COPY", ". .") in steps and steps.index(("COPY", ". .")) > names.index("RUN")
print("✅ Вправа 1 пройдена")

---
## 2. `.dockerignore`: що потрапить в образ

`docker build .` спершу пакує **контекст** — усю папку, крім того, що в `.dockerignore`. Правила Docker:

- шлях рахується **від кореня контексту**; `/` на початку й у кінці шаблону не важить;
- `*` — будь-що в межах однієї папки, `**` — будь-яка кількість папок;
- шаблон, що збігся з папкою, виключає все всередині;
- `!шаблон` — виняток: повертає файл назад;
- перевіряються **всі** рядки згори донизу, перемагає **останній**, що збігся.

`pattern_to_regex` уже написано — він перекладає шаблон у регулярний вираз.

In [ ]:
import re


def pattern_to_regex(pattern: str) -> re.Pattern[str]:
    pattern = pattern.strip("/")
    out, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            out, i = out + "(?:.*/)?", i + 3
        elif pattern.startswith("**", i):
            out, i = out + ".*", i + 2
        elif pattern[i] == "*":
            out, i = out + "[^/]*", i + 1
        elif pattern[i] == "?":
            out, i = out + "[^/]", i + 1
        elif pattern[i] == "[":
            end = pattern.index("]", i)
            out, i = out + pattern[i:end + 1], end + 1
        else:
            out, i = out + re.escape(pattern[i]), i + 1
    return re.compile(out + "(?:/.*)?$")           # збіг з папкою — це й усе в ній


print(pattern_to_regex("**/__pycache__").pattern)
print(bool(pattern_to_regex("**/__pycache__").match("news_hub/bot/__pycache__/api.pyc")))

### Вправа 2

Напиши `excluded(path, patterns)` — чи виключить Docker файл `path` (шлях від кореня, через `/`). Порожні рядки й коментарі пропускай.

🔮 **Прогноз:** старий шаблон `__pycache__/` (без `**/`). Чи виключить він `news_hub/__pycache__/api.cpython-313.pyc`?

In [ ]:
def excluded(path: str, patterns: list[str]) -> bool:
    # YOUR CODE HERE
    pass


cases = {
    ".env": True, ".env.production": True, ".env.example": False,
    ".venv/lib/python3.12/site-packages/fastapi/__init__.py": True,
    "news_hub/__pycache__/api.cpython-313.pyc": True, "news_hub/api.py": False,
    "tests/unit/test_models.py": True, "news_hub.db": True, "data/rbc_news_snapshot.json": False,
    "migrations/versions/0001_news_table.py": False, "requirements.txt": False, "requirements-dev.txt": True,
}
for path, expected in cases.items():
    got = excluded(path, DOCKERIGNORE)
    print(f"{'виключено' if got else 'в образ  '}  {path}")
    assert got == expected, path

old_rule = excluded("news_hub/__pycache__/api.cpython-313.pyc", ["__pycache__/"])
print("\nстарий шаблон __pycache__/ виключає news_hub/__pycache__/…:", old_rule)
assert old_rule is False
print("✅ Вправа 2 пройдена")

<details>
<summary>Чому <code>__pycache__/</code> не спрацював</summary>

Шаблон рахується від кореня: `__pycache__/` — це лише `./__pycache__`. Кеші в `news_hub/` і `news_hub/bot/` доїхали б в образ — і шар коду інвалідувався б після кожного запуску тестів. Правильно — `**/__pycache__`.
</details>

### Вправа 3. Контекст збірки

Пройди всі файли проєкту (`Path(".").rglob("*")`, лише файли) і збери `context` — список шляхів (через `/`), які **не** виключені. Порахуй їхній розмір у байтах (`size`).

In [ ]:
# YOUR CODE HERE

print(f"{len(context)} файлів, {size / 1024:.0f} кБ")
for path in context[:12]:
    print(" ", path)
assert "news_hub/api.py" in context and "Dockerfile" not in context
assert not any(p.startswith(("tests/", ".venv/")) or p == ".env" or "__pycache__" in p for p in context)
print("✅ Вправа 3 пройдена")

Справжній Docker на цій самій папці (урок 49, `COPY . /ctx` + `find -type f`) — 35 файлів. Для порівняння: старий Dockerfile без `.dockerignore` у папці студента з `.venv` — контекст 1.17 ГБ і 27 секунд лише на передачу.

---
## 3. Кеш шарів

Для кожної інструкції Docker шукає в кеші шар, зібраний з **тих самих вхідних даних** поверх **того самого** попереднього шару. Як тільки одна інструкція виконується заново, кеш для **всіх наступних** уже не діє.

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph S1["змінено news_hub/models.py: COPY requirements.txt"]
        direction LR
        A1["вхід: requirements.txt"] --> A2["не змінився"] --> A3["CACHED"]
    end
    subgraph S2["RUN pip install"]
        direction LR
        B1["попередній шар з кешу"] --> B2["CACHED"]
    end
    subgraph S3["COPY . ."]
        direction LR
        C1["вхід: усі файли контексту"] --> C2["models.py змінився"] --> C3["RUN: 0.1 с"]
    end
    subgraph S4["USER, ENV, CMD"]
        direction LR
        D1["попередній шар новий"] --> D2["RUN: метадані, миттєво"]
    end
    S1 --> S2 --> S3 --> S4

    class A1,B1,C1,D1 step
    class A2,C2 decision
    class A3,B2 success
    class C3,D2 warning
```

### Вправа 4

Напиши `cache_report(steps, changed)` → список `"CACHED"` / `"RUN"` для кожної інструкції з `steps` (вправа 1), якщо змінились файли `changed`. Вхідні дані:

- `COPY <файли> <куди>` — ці файли (`.` — будь-який файл контексту);
- решта інструкцій — лише сам текст інструкції (він не змінюється).

`FROM` вважай завжди `CACHED`.

In [ ]:
def cache_report(steps: list[tuple[str, str]], changed: set[str]) -> list[str]:
    # YOUR CODE HERE
    pass


def show(title: str, steps: list[tuple[str, str]], changed: set[str]) -> list[str]:
    report = cache_report(steps, changed)
    print(title)
    for (name, args), status in zip(steps, report):
        print(f"  {status:<6} {name} {args[:50]}")
    return report


good = show("правильний порядок, змінено models.py:", steps, {"news_hub/models.py"})
pip_index = [n for n, _ in steps].index("RUN")
assert good[pip_index] == "CACHED" and good[-1] == "RUN"

BAD = parse_dockerfile("""
FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
CMD ["uvicorn", "news_hub.api:app", "--host", "0.0.0.0", "--port", "8000"]
""")
bad = show("\nCOPY . . перед pip install, змінено models.py:", BAD, {"news_hub/models.py"})
assert bad[3] == "RUN"
req = show("\nправильний порядок, змінено requirements.txt:", steps, {"requirements.txt"})
assert req[pip_index] == "RUN"
print("✅ Вправа 4 пройдена")

Виміряно на `news_hub`: `COPY . .` перед `pip install` — **39.9 с** на кожну зміну коду; правильний порядок — **0.6 с**.

---
## 4. Секрети й змінні середовища

### Вправа 5. Секрет в образі

`ENV` у Dockerfile записується в метадані образу — `docker history` і `docker inspect` покажуть значення кожному. Напиши `secrets_in_dockerfile(steps)` → імена змінних з `ENV` чи `ARG`, у назві яких є `KEY`, `SECRET`, `TOKEN` або `PASSWORD` і яким задано значення. `ENV` буває в двох формах: `ENV A=1 B=2` і `ENV A 1`.

In [ ]:
SECRET_WORDS = ("KEY", "SECRET", "TOKEN", "PASSWORD")


def secrets_in_dockerfile(steps: list[tuple[str, str]]) -> list[str]:
    # YOUR CODE HERE
    pass


print("news_hub:", secrets_in_dockerfile(steps))
assert secrets_in_dockerfile(steps) == []
leaky = parse_dockerfile("ENV PYTHONUNBUFFERED=1 GEMINI_API_KEY=AIza-demo\nENV BOT_TOKEN 123:ABC\nARG PIP_INDEX_URL\nENV DEBUG=0")
print("leaky:", secrets_in_dockerfile(leaky))
assert secrets_in_dockerfile(leaky) == ["GEMINI_API_KEY", "BOT_TOKEN"]
print("✅ Вправа 5 пройдена")

### Вправа 6. `--env-file` бере значення буквально

`python -m news_hub.security` друкує рядки для `export` у shell — з одинарними лапками, бо в bcrypt-хеші є `$`. Compose знімає лапки, а `docker run --env-file` — ні.

Напиши два читачі `.env`: `read_docker(text)` (значення як є) і `read_compose(text)` (якщо значення в однакових лапках `'…'` чи `"…"` — зняти їх). Потім перевір правило `news_hub` (урок 47): хеш пароля має починатися з `$2`.

🔮 **Прогноз:** з яким читачем застосунок стартує?

In [ ]:
ENV_TEXT = """
# рядки від python -m news_hub.security
ADMIN_PASSWORD_HASH='$2b$12$eBDJbnUjhmlfW0bqlm9n1uCzKlO6dQoHzU6o1v2Ej2m4kB7l1Qv6y'
JWT_SECRET='4GnN8DU8jRhAk6z09C-avMa3mSJd'
"""


def read_docker(text: str) -> dict[str, str]:
    # YOUR CODE HERE
    pass


def read_compose(text: str) -> dict[str, str]:
    # YOUR CODE HERE
    pass


def app_starts(env: dict[str, str]) -> bool:
    return env["ADMIN_PASSWORD_HASH"].startswith("$2")        # спрощене правило load_admin_settings


print("docker run --env-file:", app_starts(read_docker(ENV_TEXT)), read_docker(ENV_TEXT)["ADMIN_PASSWORD_HASH"][:12])
print("compose env_file:     ", app_starts(read_compose(ENV_TEXT)), read_compose(ENV_TEXT)["ADMIN_PASSWORD_HASH"][:12])
assert not app_starts(read_docker(ENV_TEXT)) and app_starts(read_compose(ENV_TEXT))
print("✅ Вправа 6 пройдена")

Справжній запуск (урок 49): з лапками — `RuntimeError: ADMIN_PASSWORD_HASH — bcrypt-хеш ($2b$…), а не пароль`, контейнер `Exited (3)`. Для `--env-file` — `python -m news_hub.security --env-file` (без лапок).

---
## 5. Сигнали: чому `docker stop` чекає 10 секунд

`docker stop` надсилає `SIGTERM` процесу з PID 1 і чекає 10 с, а потім — `SIGKILL`. Shell-форма `CMD uvicorn …` робить PID 1 оболонку `/bin/sh -c`, а exec-форма `CMD ["uvicorn", …]` — сам застосунок. Те саме відтворимо без Docker: «сервер» ловить `SIGTERM` і пише в журнал, що закриває з'єднання.

In [ ]:
import signal
import sys
import textwrap
import time

Path("server.py").write_text(textwrap.dedent("""
    import os, signal, sys, time
    from pathlib import Path
    Path("server.pid").write_text(str(os.getpid()))
    def stop(signum, frame):
        with open("server.log", "a") as log:
            log.write("SIGTERM: закриваю з'єднання\\n")
        sys.exit(0)
    signal.signal(signal.SIGTERM, stop)
    while True:
        time.sleep(0.1)
"""))


def alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except ProcessLookupError:
        return False


def stop_like_docker(command: list[str]) -> dict[str, object]:
    """Запустити command, надіслати SIGTERM першому процесу (як docker stop — PID 1), подивитись, що сталось."""
    for name in ("server.pid", "server.log"):
        Path(name).unlink(missing_ok=True)
    top = subprocess.Popen(command)
    while not Path("server.pid").exists():
        time.sleep(0.05)
    server = int(Path("server.pid").read_text())
    top.send_signal(signal.SIGTERM)
    top.wait(timeout=5)
    time.sleep(0.5)
    survived = alive(server)
    if survived:
        os.kill(server, signal.SIGKILL)            # як docker через 10 с
    log = Path("server.log").read_text().strip() if Path("server.log").exists() else ""
    return {"сервер = перший процес": top.pid == server, "журнал": log, "сервер пережив SIGTERM": survived}

### Вправа 7

Порівняй три запуски «сервера»: напряму (`[sys.executable, "server.py"]`), через оболонку (`["sh", "-c", "<python> server.py"]`) і через оболонку з `exec`. Збережи результати в `direct`, `shell`, `with_exec`.

🔮 **Прогноз:** у якому випадку в журналі не буде рядка про закриття з'єднань?

In [ ]:
# YOUR CODE HERE

for title, result in (("напряму (exec-форма CMD)", direct), ("sh -c (shell-форма CMD)", shell),
                      ("sh -c exec …", with_exec)):
    print(f"{title:<26} {result}")
assert direct["журнал"] and with_exec["журнал"] and not shell["журнал"] and shell["сервер пережив SIGTERM"]
print("✅ Вправа 7 пройдена")

<details>
<summary>Що сталось</summary>

З `sh -c` сигнал отримала оболонка і завершилась, а сервер навіть не дізнався про зупинку — у контейнері Docker через 10 с убив би все `SIGKILL` (код 137), і lifespan FastAPI не виконався б. `exec` замінює оболонку на процес Python: той стає першим і отримує сигнал сам. Саме це робить exec-форма `CMD ["uvicorn", …]`.

Справжній `docker stop` для `news_hub` (урок 49): exec-форма — 1.8 с, код 0, «Application shutdown complete»; shell-форма — 10.2 с, код 137.
</details>

---
## 6. Знайди помилку

Dockerfile нижче збирається, і `curl /health` відповідає `{"status":"ok"}`. Але в ньому три проблеми.

### Вправа 8

Напиши `lint(text)` → множину кодів проблем:

- `"order"` — `COPY . .` стоїть перед `RUN pip install` (використай `cache_report`: що буде з `pip install` після зміни `news_hub/models.py`?);
- `"secret"` — секрет у `ENV`/`ARG` (`secrets_in_dockerfile`);
- `"shell-cmd"` — `CMD` не в exec-формі (не починається з `[`).

На Dockerfile проєкту `lint` має повернути порожню множину.

In [ ]:
FIND_BUG = """
FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
RUN useradd --uid 10001 app
USER app
ENV GEMINI_API_KEY=AIzaSyD-course-demo-key-not-real-0000000
CMD uvicorn news_hub.api:app --host 0.0.0.0 --port 8000
"""


def lint(text: str) -> set[str]:
    # YOUR CODE HERE
    pass


print("find-bug:", sorted(lint(FIND_BUG)))
print("news_hub:", sorted(lint(DOCKERFILE)))
assert lint(FIND_BUG) == {"order", "secret", "shell-cmd"} and lint(DOCKERFILE) == set()
print("✅ Вправа 8 пройдена")

<details>
<summary>Відповідь і справжній вивід</summary>

```text
$ echo "# x" >> news_hub/models.py && time docker build -q -t findbug .
real    0m43.050s                       ← order: pip install на кожну зміну коду

$ docker history findbug --format '{{.CreatedBy}}' | grep GEMINI
ENV GEMINI_API_KEY=AIzaSyD-course-demo-key-n…    ← secret: ключ бачить кожен, хто має образ

$ time docker stop fb
real    0m10.262s
$ docker inspect -f '{{.State.ExitCode}}' fb
137                                     ← shell-cmd: SIGTERM не дійшов до uvicorn
```

`curl /health` жодної з цих проблем не показує: вони не в тому, **чи** працює застосунок, а в тому, як його збирають, що в ньому лежить і як він зупиняється. У проєкті це перевіряють тести `tests/docker/test_image.py` (`pytest -m docker`).
</details>

In [ ]:
for name in ("server.py", "server.pid", "server.log"):
    Path(name).unlink(missing_ok=True)
print("готово")

---
## Локально, з Docker

```bash
cd module_5/lessons/lesson_49_docker/news_hub
docker build -t news_hub:48 .
docker network create news
docker run -d --name redis --network news redis:7-alpine
docker run --rm -v news_data:/data news_hub:48 alembic upgrade head
docker run -d --name api --network news -p 8000:8000 -v news_data:/data \\
    -e REDIS_URL=redis://redis:6379/0 --env-file .env news_hub:48
curl localhost:8000/health/ready
docker stop api
pytest -m docker            # 6 тестів образу
```

```text
{"status":"ok","checks":{"database":"ok","redis":"ok"}}
```

## Самоперевірка

1. Чому `.env` не можна прибрати з образу командою `RUN rm .env`?
2. Змінився `requirements.txt`. Які інструкції виконаються заново?
3. Що означає `localhost` усередині контейнера?
4. Чим `/health` відрізняється від `/health/ready`?
5. Що означає код виходу 137?

## Далі

- Книга курсу: [Урок 49](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_48/) — старий Dockerfile проти нового з реальними числами, томи, мережа, здоров'я і зупинка, архітектура.
- У папці `news_hub`: `pytest -m docker`.
- Урок 50 — Compose: API, міграції, PostgreSQL, Redis, nginx і бот однією командою; деплой.